# Geocoding

The following section breaks down the process of geocoding violation location address and respondent home address in the following steps:

**1. Building and Cleaning Addresses:** This section uses various fields from `violations_df_full` to build violation and respondent addresses appropriately, then clean them before geocoding

**2. Saving Intersections for Separate Geocoding:** This section creates dataframes `violation_intersections.csv` and `respondent_intersections.csv` for addresses that are intersections. These will require separate geocoding

**3. Running Addresses through NYC GeoSupport Geocoder:** This section uses async batching to send multiple requests, reducing runtime.

**4. Flagging Levels of Geocoding Precision:**
This section adds the following tier categorizations for the geocoded addresses:

`0` if no match
`1` for an exact match
`2` for a fallback match on the same street
`3` for a fallback match in the same neighborhood
`4` for a fallback match with the same ZIP code
`5` for a fallback match in the same borough
`nan` if the matched address is something else

In [18]:
DATA_DIR = "/Users/amelie/Desktop/GitHub/nyc-street-vendor-enforcement/data/processed"

In [ ]:
# Setup

import pandas as pd
import sys, os
sys.path.append(os.path.abspath("..")) 

#DATA_DIR = "...data/processed"
os.makedirs(DATA_DIR, exist_ok=True)

#Load df and modify datetime
violations_df_full = pd.read_csv(f"{DATA_DIR}/violations_df_full.csv")
violations_df_full['violation_date'] = pd.to_datetime(violations_df_full['violation_date'], errors='coerce')
violations_df_full['violation_year'] = violations_df_full['violation_date'].dt.year

In [21]:
violations_df_full.shape

(109567, 36)

## Building and Cleaning Address Strings